# Prediction driver

Fine-tuned and optional cloud cells use the shared run matrix. Model, diarization and API runs default off; the checkpoint and provider allowances require explicit values.


In [ ]:
import subprocess
import sys

INSTALL_RUNTIME = False
if INSTALL_RUNTIME:
    subprocess.run([sys.executable, '-m', 'pip', 'install',
                    'transformers>=4.46.3,<5', 'faster-whisper==1.2.1',
                    'ctranslate2>=4.4,<5', 'pyannote.audio>=4,<5',
                    'librosa', 'soundfile', 'google-cloud-speech>=2.27,<3',
                    'deepgram-sdk>=7,<8'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e',
                    '/kaggle/working/shono'], check=True)


In [ ]:
import os
from pathlib import Path

from shono.data import LicenseRegistry, Manifest
from shono.protocol import execute_matrix, run_matrix
from shono.provenance import RunContext

REPO = Path('/kaggle/working/shono')
AUDIO_ROOT = Path('/kaggle/input')
FT_MODEL = 'unknown'  # Supply an actual trained checkpoint explicitly.
RUN_MODELS = False
RUN_RTF = False
RUN_DIARIZATION = False
RUN_APIS = False
ALLOW_PAID = False
PROVIDER_ALLOWANCES = {'google': 0.0, 'deepgram': 0.0}
registry = LicenseRegistry.load(REPO / 'data/licenses.json')
entries = run_matrix(fine_tuned_model=FT_MODEL)

from shono.transcribe import (
    FasterWhisperTranscriber, LongFormPipelineTranscriber, LongFormTranscriber,
    ShortFormWhisperTranscriber, SileroVAD,
)
from shono.transcribe.convert import to_ct2

CT2_ROOT = Path('/kaggle/working/shono-ct2')
LOCAL_CHECKPOINTS = {}  # Explicit slug -> downloaded local Whisper checkpoint.
CONVERTED_MODELS = {}


def local_transcriber(entry):
    if entry.inference_mode == 'bounded-short':
        return ShortFormWhisperTranscriber(entry.model_id, language='bn', device='cuda')
    checkpoint = LOCAL_CHECKPOINTS.get(entry.slug)
    if checkpoint is None and entry.slug == 'shono-medium':
        checkpoint = entry.model_id
    if checkpoint is None or not Path(checkpoint).is_dir():
        raise ValueError('Declare an existing local checkpoint for ' + entry.slug)
    converted = CT2_ROOT / entry.slug
    key = (entry.model_id, str(Path(checkpoint).resolve()))
    if key not in CONVERTED_MODELS:
        # Rebuild scratch conversion once per checkpoint in this notebook run.
        to_ct2(checkpoint, converted, quantization='float16', force=True)
        CONVERTED_MODELS[key] = str(converted)
    pipeline = LongFormTranscriber(
        SileroVAD(), FasterWhisperTranscriber(str(converted), language='bn', device='cuda'),
        max_chunk_s=30.0, pad_s=0.25,
    )
    return LongFormPipelineTranscriber(pipeline)


## Local predictions

The fine-tuned system covers every applicable read, code-switch and whole-recording cell. Optional private slices remain pending when absent. Source manifests are input definitions and are not rewritten here.


In [ ]:
our_entries = tuple(entry for entry in entries if entry.slug == 'shono-medium')
outcomes = execute_matrix(
    our_entries, base_dir=REPO, audio_root=AUDIO_ROOT, registry=registry,
    transcriber_for=local_transcriber, run_models=RUN_MODELS,
    inference_settings={
        'shono-medium:bounded-short': {'device': 'cuda', 'language': 'bn',
                                       'task': 'transcribe', 'max_window_s': 30.0},
        'shono-medium:long-form-vad': {'device': 'cuda', 'language': 'bn',
            'max_chunk_s': 30.0, 'pad_s': 0.25, 'quantization': 'float16',
            'compute_type': 'float16', 'local_checkpoint': str(FT_MODEL),
            'condition_on_previous_text': False, 'word_timestamps': True,
            'vad_threshold': 0.5, 'vad_min_speech_ms': 250}},
)
for outcome in outcomes:
    print(outcome.entry.slice_name, outcome.status, outcome.reason)


In [ ]:
# An optional CLI run measures actual elapsed time/audio duration.
if RUN_RTF:
    recording = Manifest.from_jsonl(REPO / 'data/bengali-loop-test.manifest.jsonl')
    recording.validate_against(registry)
    from shono.data.audio_paths import validate_audio_paths
    paths = validate_audio_paths(recording, AUDIO_ROOT)
    sample = paths[0]
    subprocess.run([sys.executable, '-m', 'shono.transcribe', str(sample),
                    '--model', str(CT2_ROOT / 'shono-medium')], cwd=REPO, check=True)


## Optional diarization

The published Loop CSV parser reads explicit clock boundaries. DER uses a 0.25-second collar with overlap included. Source permission must be confirmed before this path can run.


In [ ]:
if RUN_DIARIZATION:
    from shono.diarize import (DERConfig, PyannoteDiarizer, corpus_der,
                              der_bootstrap_ci, load_loop_csv,
                              render_der_report, vad_intersection)
    if registry.require('bengali_loop').status == 'excluded':
        raise ValueError('Bengali-Loop source permission is unconfirmed')
    diar_root = Path('/kaggle/input/bengali-loop/diarization')
    references = sorted(diar_root.glob('*.csv'))
    if not references:
        raise ValueError('No declared diarization reference CSVs')
    diarizer = PyannoteDiarizer(hf_token=os.environ.get('HF_TOKEN'), device='cuda')
    vad = SileroVAD()
    pairs = []
    for reference in references:
        audio = diar_root / (reference.stem + '.wav')
        if not audio.is_file():
            raise ValueError(f'Missing audio for {reference.name}')
        hypothesis = vad_intersection(diarizer.diarize(str(audio)), vad.detect(str(audio)))
        pairs.append((load_loop_csv(reference), hypothesis))
    config = DERConfig(collar_s=0.25, skip_overlap=False)
    result = corpus_der(pairs, config)
    interval = der_bootstrap_ci(pairs, config, seed=0) if len(pairs) >= 2 else None
    context = RunContext.capture(0, {'collar_s': 0.25, 'skip_overlap': False},
                                 command='predict.ipynb diarization', repo=REPO)
    text = render_der_report([('pyannote community-1 + VAD', result, interval)],
                             config, generated=context.timestamp)
    output = REPO / 'outputs/diarization.md'
    output.parent.mkdir(parents=True, exist_ok=True)
    output.write_text(text, encoding='utf-8')


## Code-switch report

The local matrix already includes MUCS and the optional mini-eval. These cells use script normalization v1.0.0, separately from Bengali v1.1.0. The shipped four-row MUCS example is not a production input.


In [ ]:
subprocess.run([sys.executable, '-m', 'shono.eval', '--report', 'codeswitch'],
               cwd=REPO, check=True)


## Optional cloud predictions

Separate declared provider allowances default to zero. The full planned workload is checked before any adapter is created and each run consumes the same provider guard. Account credit, retries and billing remain unknown. Google long-form stays pending.


In [ ]:
from shono.api import DeepgramTranscriber, GoogleChirpTranscriber


def cloud_transcriber(entry):
    if entry.provider == 'google':
        return GoogleChirpTranscriber(project_id=os.environ.get('GCP_PROJECT', ''))
    return DeepgramTranscriber(api_key=os.environ.get('DEEPGRAM_KEY', ''))


cloud_entries = tuple(entry for entry in entries if entry.provider != 'local')
outcomes = execute_matrix(
    cloud_entries, base_dir=REPO, audio_root=AUDIO_ROOT, registry=registry,
    transcriber_for=cloud_transcriber, run_apis=RUN_APIS,
    allowances=PROVIDER_ALLOWANCES, allow_paid=ALLOW_PAID,
    inference_settings={'google-chirp': {'language': 'bn-BD', 'region': 'us-central1',
                                        'decoding': 'Recognize chirp_2; retry disabled'},
                        'deepgram-nova3': {'language': 'bn', 'smart_format': True,
                                          'decoding': 'Nova-3; retry disabled'}},
)
for outcome in outcomes:
    print(outcome.entry.system, outcome.entry.slice_name, outcome.status, outcome.reason)
for name in ['longform', 'codeswitch', 'full']:
    subprocess.run([sys.executable, '-m', 'shono.eval', '--report', name],
                   cwd=REPO, check=True)
